# Reload work and retry publication

## Objective

Reopen a saved run and understand when to reload, retry or train again.

Run notebook 4 first. Loading checks saved evidence without fitting again. Keep all referenced study and tuning folders, not just the final workflow folder.

An ordinary `run_study` call creates a new run. Campaigns can resume matching work. Retrying publication writes the saved evidence and never trains again.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


## Step 2: Open the saved run

We will load the workflow created in notebook 4 using its saved path.


In [ ]:
# Import the tools used in this step.
from autoforecast.study_workflow import load_forecast_workflow

# Check and open the saved run without fitting again.
result = load_forecast_workflow((work / "latest-run.txt").read_text().strip())

### Understand the output

No output is expected. We have reopened the run using the path saved in notebook 4.


## Step 3: Check that loading preserves the result

We will open the same run a second time and check that its path and validation scores agree. No model fit is needed.


In [ ]:
# Remember the workflow folder we already opened.
before = result.path
# Open it again and compare the saved identity and validation scores.
again = load_forecast_workflow(before)
assert again.path == before
assert again.validation_metrics == result.validation_metrics
# Print only the identity and score needed for this check.
print("Same run:", again.run_id)
print("Validation MAE:", again.validation_metrics["mae"])

### Understand the output

Both assertions should pass. You should see the saved run ID and the same validation MAE as before, about 2.83 for the unchanged tuned example. No fit was needed. This does not mean a new training call reuses the workflow; an ordinary new workflow call creates a new run.


To publish later, choose one destination in notebooks 7, 8 or 9. Calling `registry.publish_results` again with the same run and destination checks and reuses identical rows. Different rows under the same identity are refused.

For more models, `Study.run_campaign` accepts `StudyModel` entries with importable callbacks. It runs sequentially by default. Local parallel workers need accessible model code and shared paths; cluster execution also needs an existing Ray cluster. It does not distribute a custom model's internal training algorithm.

To reuse chosen settings on another registered study, use `Study.run_forecast_workflow` with `selected_config=result.selected_config`, the same implementation identity and no new fixed parameters. This performs new fits on the new study. Keep the model's source and profile unchanged when claiming the same implementation.

Source retention records files, not proof of every import, executed notebook cell or hidden object state. Store settings in the profile and select all relevant helpers.
